# BEAUTYPRICE INTELLIGENCE — EDA
Notebook riêng cho Nhiệm vụ 1, Giai đoạn 6. Chạy Run All để đọc CSV thật, tạo lại `01_ket_qua_eda.xlsx` và hiển thị kết quả.

In [1]:
from pathlib import Path
import runpy, sys, pandas as pd
candidates=[Path.cwd(),Path.cwd()/'BDr'/'kqptr',Path.cwd()/'kqptr']
folder=next(p for p in candidates if (p/'01_eda.py').is_file())
sys.path.insert(0,str(folder.resolve()))
g=runpy.run_path(str(folder/'01_eda.py'))

Created: C:\Users\Latitude 5500\OneDrive\Big Data\BDr\kqptr\01_ket_qua_eda.xlsx
Giá hợp lệ: 6,350; min=5,000; max=3,160,000; ngoại lệ IQR=314


## Tập dữ liệu, số quan sát và điều kiện lọc
Một dòng `price_snapshots` là một quan sát. Giá hợp lệ là số hữu hạn lớn hơn 0. Giá trống, sai kiểu, bằng 0 hoặc âm bị loại khỏi thống kê giá nhưng vẫn nằm trong mẫu số chất lượng.

In [2]:
pd.DataFrame(g['overview_rows'],columns=['Chỉ tiêu','Giá trị'])

Chỉ tiêu,Giá trị
Tập dữ liệu chính,price_snapshots (1).csv
Đơn vị quan sát,Một dòng snapshot giá
Phạm vi thời gian,2026-10-18 đến 2026-10-21 (UTC+7)
Tổng snapshot,6350
Sản phẩm duy nhất trong products,5657
Thương hiệu có giá trị,442
Danh mục trong bảng liên kết,6
Nhà bán lẻ,5
Giá hiện tại hợp lệ,6350
Ngày thu thập thiếu/sai,0


## Thống kê mô tả
Bảng có số quan sát, tỷ lệ thiếu/không hợp lệ, giá thấp nhất, giá cao nhất, trung bình, trung vị, độ lệch chuẩn, Q1, Q3 và IQR.

In [3]:
g['numeric_frame']

variable,source,total_n,valid_n,missing_n,invalid_n,missing_invalid_rate_pct,min,q1,mean,median,q3,max,std,iqr,lower_bound,upper_bound,outlier_candidate_n,outlier_candidate_rate_pct,unit,filter_conditions
current_price_vnd,price_snapshots (1).csv,6350,6350,0,0,0.000000,5000.0,145000.000000,316861.906142,244000.000000,406000.000,3.160000e+06,278083.631209,261000.000000,-246500.0,797500.0,314.0,4.944882,VND,Finite current price > 0; IQR candidates retained
listed_price_vnd,price_snapshots (1).csv,6350,4861,1489,0,23.448819,16000.0,202000.000000,440679.680724,350000.000000,555000.000,3.950000e+06,369924.852311,353000.000000,NaN,NaN,NaN,NaN,VND,Finite listed price > 0
discount_percent,price_snapshots (1).csv,6350,4861,1489,0,23.448819,0.0,15.060000,27.768655,24.980000,38.000,9.816000e+01,16.129147,22.940000,NaN,NaN,NaN,NaN,%,"Resolved input discount in [0,100]; original website value is discount_percent_source"
discount_percent_calculated,price_snapshots (1).csv,6350,4861,1489,0,23.448819,0.0,15.100000,27.771401,24.930000,38.120,9.816000e+01,16.126708,23.020000,NaN,NaN,NaN,NaN,%,"Source calculated discount in [0,100]"
discount_percent_recalculated,price_snapshots (1).csv,6350,4861,1489,0,23.448819,0.0,15.102041,27.771416,24.926254,38.125,9.815951e+01,16.126671,23.022959,NaN,NaN,NaN,NaN,%,P>0; L>0; L>=P; exclude anomalous negative discounts
rating,hasaki_reviews.csv + guardian_reviews.csv,6673,6673,0,0,0.000000,2.5,5.000000,4.899820,5.000000,5.000,5.000000e+00,0.301321,0.000000,NaN,NaN,NaN,NaN,stars,All source review rows; range check 1–5; half stars retained


## Chất lượng dữ liệu
Không chỉ khảo sát giá: notebook thống kê giá trị trống theo mọi trường chính và các trạng thái thu thập, liên kết, giá và khuyến mãi.

In [4]:
display(g['categorical_frame'])
display(g['coverage_frame'])
display(pd.DataFrame(g['grouped_rows'],columns=g['group_headers']))

source,variable,value,count,percentage,unique_count,denominator_n
price_snapshots,brand,Cocoon,350,5.511811,442,6350
price_snapshots,brand,EUCERIN,192,3.023622,442,6350
price_snapshots,brand,Garnier,175,2.755906,442,6350
price_snapshots,brand,DOVE,146,2.299213,442,6350
price_snapshots,brand,SUNPLAY,143,2.251969,442,6350
price_snapshots,brand,La Roche-Posay,121,1.905512,442,6350
price_snapshots,brand,BIODERMA,111,1.748031,442,6350
price_snapshots,brand,Vichy,108,1.700787,442,6350
price_snapshots,brand,OLAY,104,1.637795,442,6350
price_snapshots,brand,Hada Labo,104,1.637795,442,6350


scope,metric,value,definition
PRICE,total_price_snapshots,6350,All input snapshot rows
PRICE,unique_product_id,5657,Unique snapshot products
PRICE,master_unique_products,5657,Products master; distinct from snapshot products
PRICE,unique_brands,442,Nonblank snapshot brand strings; no alias normalization
PRICE,unique_primary_categories,6,Nonblank primary_category; missing shown separately
PRICE,unique_linked_categories,6,Distinct labels in category relationship table
PRICE,unique_retailers,5,Distinct retailers observed
PRICE,earliest_collected_at,2026-10-18 07:09:23+07:00,UTC+7
PRICE,latest_collected_at,2026-10-21 19:05:49+07:00,UTC+7
PRICE,observed_dates,4,Distinct observed local dates


Nhóm,Giá trị,Snapshot,Giá hợp lệ,Thiếu/sai,Tỷ lệ thiếu/sai,Min (VND),Max (VND),Trung bình,Trung vị,Q1,Q3,IQR,Ứng viên ngoại lệ
retailer_name,Cocolux,734,734,0,0.0,5000.0,3160000.0,3.848012e+05,299000.0,179250.00,492250.00,313000.00,35
retailer_name,Guardian,994,994,0,0.0,31000.0,3160000.0,3.228204e+05,247500.0,152000.00,412750.00,260750.00,43
retailer_name,Hasaki,1871,1871,0,0.0,9000.0,2625000.0,2.964594e+05,227000.0,132500.00,360000.00,227500.00,135
retailer_name,The Gioi Skin Food,500,500,0,0.0,24000.0,1409000.0,2.791000e+05,231500.0,136750.00,376500.00,239750.00,16
retailer_name,Watsons,2251,2251,0,0.0,17000.0,2963000.0,3.174234e+05,249000.0,144000.00,418000.00,274000.00,90
primary_category,Kem chống nắng,1049,1049,0,0.0,9000.0,2625000.0,3.234820e+05,289000.0,169000.00,431000.00,262000.00,20
primary_category,Serum,1121,1121,0,0.0,20000.0,3160000.0,4.906780e+05,397000.0,240000.00,598000.00,358000.00,77
primary_category,Sữa rửa mặt,1043,1043,0,0.0,9000.0,1632000.0,2.391663e+05,179000.0,99000.00,340000.00,241000.00,29
primary_category,Sữa tắm,942,942,0,0.0,9000.0,2686000.0,2.224826e+05,186000.0,135000.00,260000.00,125000.00,64
primary_category,Tẩy trang,640,640,0,0.0,5000.0,1449000.0,2.394555e+05,195000.0,116500.00,322250.00,205750.00,22


## Ngoại lệ và cách xử lý
Ứng viên nằm ngoài Q1−1,5×IQR hoặc Q3+1,5×IQR chỉ được đánh dấu. Không tự xóa hoặc sửa vì mức giá có thể khác do thương hiệu, dung tích, quy cách hoặc phân khúc. CSV nguồn được giữ nguyên.

In [5]:
print('Ứng viên ngoại lệ:',len(g['outlier_rows']))
pd.DataFrame(g['outlier_rows'],columns=['snapshot_id','product_id','retailer','danh mục','thương hiệu','collected_at','current_price_vnd','listed_price_vnd','source_file']).head(20)

Ứng viên ngoại lệ: 314


snapshot_id,product_id,retailer,danh mục,thương hiệu,collected_at,current_price_vnd,listed_price_vnd,source_file
snap_3d2bdacfb8eb621a,prod_063773a059be6c88,Cocolux,Serum,LUVUM,2026-10-21T08:13:42+07:00,858000,NaN,cocolux_serum_page_4.html
snap_ebbf45c33e7b80a5,prod_09e3d594f5453049,Cocolux,Tẩy tế bào chết,OBAGI,2026-10-21T08:49:39+07:00,960000,NaN,cocolux_exfoliators_page_2.html
snap_a5dae9a4055c7125,prod_0c57fdc664fad36c,Cocolux,Serum,SVR,2026-10-21T08:13:42+07:00,989000,1099000.0,cocolux_serum_page_4.html
snap_f965599aac5d59be,prod_1e41a928ba8a87c4,Cocolux,Serum,oh!oh!,2026-10-21T08:13:42+07:00,799000,980000.0,cocolux_serum_page_4.html
snap_6cf36f0f0c64d511,prod_1f91dab13d8e80cc,Cocolux,Kem chống nắng,Martiderm,2026-10-21T07:33:02+07:00,799000,1380000.0,cocolux_sunscreens_page_4.html
snap_d8de2429a6c2da76,prod_1fb73102de10798a,Cocolux,Kem chống nắng,Martiderm,2026-10-21T07:33:02+07:00,799000,1380000.0,cocolux_sunscreens_page_4.html
snap_18be95ac82913c34,prod_22ddd5c4db44a178,Cocolux,Serum,SVR,2026-10-21T08:13:42+07:00,999000,1119000.0,cocolux_serum_page_4.html
snap_54c63db823ae4c78,prod_24556a95c97aeeca,Cocolux,Serum,PAULA'S CHOICE,2026-10-21T08:13:42+07:00,1640000,2050000.0,cocolux_serum_page_4.html
snap_ac47149773a4adc5,prod_31877a45738ce914,Cocolux,Serum,OBAGI,2026-10-21T08:13:42+07:00,2338000,2750000.0,cocolux_serum_page_4.html
snap_3dbdcc0cafc08a25,prod_323a89293b9310e8,Cocolux,Serum,La Roche-Posay,2026-10-21T08:13:42+07:00,1550000,NaN,cocolux_serum_page_4.html


## Review Hasaki và Guardian độc lập
Đọc cả hasaki_reviews.csv và guardian_reviews.csv, giữ nguyên dòng nguồn. Thống kê tổng và từng nguồn; sản phẩm đếm theo cặp source + source_product_id để tránh trùng mã giữa nhà bán lẻ. Rating kiểm tra 1–5, giữ nửa sao. Ngày Hasaki kèm giờ và ngày Guardian tiếng Việt được đọc theo định dạng riêng, không dùng làm collected_at. Giữ nhãn category nguồn; cờ product_id_match chưa xác minh liên kết với bảng giá.
Bảng chất lượng theo source giữ tất cả dòng, phân biệt missing và invalid; duplicate là số dòng dư theo source + source_product_id + review_id. Source product-match flag: this source field is NOT a verified link to price products. Các giá trị và số lượng thực tế nằm trong bảng categorical theo từng nguồn. Rating là observed rating distribution in the collected sample; không suy rộng toàn bộ khách hàng. Tỷ trọng nguồn có trong sample_share; pooled metrics không phải so sánh cân bằng hai retailer, cần xem per-source metrics.


In [6]:
display(pd.DataFrame(g['review_summary_rows']).rename(columns={'unknown':'source_flag_unknown_n','unknown_rate':'source_flag_unknown_rate'}))
display(g['review_source_quality_frame'])
display(g['review_quality'])
print('Review date:',g['reviews']['review_date_parsed'].min(),g['reviews']['review_date_parsed'].max())
p=g['price'][g['valid_price']]
print(f'Giá hiện tại: n={len(p)}, min={p.min():,.0f} VND, max={p.max():,.0f} VND, mean={p.mean():,.2f} VND, median={p.median():,.0f} VND.')
print('Ứng viên IQR được giữ nguyên:',int(g['outlier_flag'].sum()))
print('Anomalous discount L<P:',int(g['anomalous_discount'].sum()))

Review date: 2017-01-11 03:50:11 2026-10-21 00:00:00
Giá hiện tại: n=6350, min=5,000 VND, max=3,160,000 VND, mean=316,861.91 VND, median=244,000 VND.
Ứng viên IQR được giữ nguyên: 314
Anomalous discount L<P: 0


source,total,products,mean,median,purchased,images,source_flag_unknown_n,source_flag_unknown_rate,sample_share
all,6673,749,4.899820,5.0,0.744942,0.161696,410,0.061442,1.000000
guardian,410,193,4.928049,5.0,1.000000,0.109756,410,1.000000,0.061442
hasaki,6263,556,4.897972,5.0,0.728245,0.165097,0,0.000000,0.938558


source,total_reviews,unique_source_products,valid_rating_n,missing_rating_n,invalid_rating_n,min_rating,max_rating,mean_rating,median_rating,missing_review_text_n,missing_review_date_n,invalid_review_date_n,duplicate_review_n,is_bought_known_n,is_bought_true_n,has_images_known_n,has_images_true_n,product_id_match_counts,sample_share
guardian,410,193,410,0,0,2.5,5.0,4.928049,5.0,0,0,0,0,410,410,410,45,"{""unknown"": 410}",0.061442
hasaki,6263,556,6263,0,0,4.0,5.0,4.897972,5.0,0,0,0,0,6263,4561,6263,1034,"{""match"": 6263}",0.938558


source,variable,value,count,percentage,unique_count,denominator_n
guardian_reviews,content_quality,usable_candidate,338,82.439024,4,410
guardian_reviews,content_quality,short,51,12.439024,4,410
guardian_reviews,content_quality,low_information,20,4.878049,4,410
guardian_reviews,content_quality,selling_or_spam,1,0.243902,4,410
guardian_reviews,is_bought,yes,410,100.000000,1,410
guardian_reviews,has_images,False,365,89.024390,2,410
guardian_reviews,has_images,True,45,10.975610,2,410
guardian_reviews,product_id_match,unknown,410,100.000000,1,410
guardian_reviews,category,Sữa rửa mặt,113,27.560976,6,410
guardian_reviews,category,Tẩy trang,82,20.000000,6,410


## Nhận xét và giới hạn
Kết quả chỉ phản ánh các nhà bán lẻ và thời gian trong dữ liệu. Thống kê theo retailer hoặc danh mục mô tả observed catalog/snapshot price level; chưa chứng minh nơi bán rẻ hơn cho cùng sản phẩm. Chưa có verified equivalent match, chưa có mẫu số matching chính thức. Review Hasaki và Guardian được phân tích độc lập và không dùng để suy luận quan hệ giá–rating. Ngoại lệ giữ lại để kiểm tra thương hiệu, dung tích và quy cách; không tự loại bỏ.